In [ ]:
import os
import pathlib
import pandas as pd

In [ ]:
root_path = pathlib.Path(os.getcwd())
root_path = root_path.parents[1]

In [ ]:
dataframe_product_detail = pd.read_excel(
    os.path.join(
        root_path,
        "data_folder",
        "product_detail_export.xlsx",
    )
)

In [ ]:
dataframe_product_detail

In [ ]:
dataframe_product_detail.dtypes

In [ ]:
dataframe_product_detail = dataframe_product_detail.rename(
    columns={
        "DISTRIBUTEUR": "distributor",
        "SOURCE DONNEES": "data_source",
        "CODE PRODUIT": "product_code",
        "DESCRIPTION": "description",
        "MARQUE": "brand",
        "INDUSTRIEL": "industrial",
        "UF": "unit",
        "Qté Facture": "quantity",
        "Montant HT": "amount_ht"
    }
)

In [ ]:
dataframe_product_detail

In [ ]:
dataframe_product_detail_distributor = dataframe_product_detail["distributor"].isna().sum()
dataframe_product_detail_distributor

In [ ]:
dataframe_product_detail.dropna(how="all", inplace=True)
dataframe_product_detail

In [ ]:
dataframe_product_detail.drop(index=719, inplace=True)
dataframe_product_detail

In [ ]:
dataframe_product_detail_filtered = dataframe_product_detail[~dataframe_product_detail["data_source"].str.contains("DECLARATIF DISTRIBUTEUR")]
dataframe_product_detail_filtered["data_source"] = dataframe_product_detail_filtered["data_source"].str.replace("AUTRES - ", "")
dataframe_product_detail_filtered

In [ ]:
# Detect lines with "DECLARATIF DISTRIBUTEUR" in the "data_source" column
dataframe_product_detail_declaratif_distributeur = dataframe_product_detail[dataframe_product_detail["data_source"].str.contains("DECLARATIF DISTRIBUTEUR")]
dataframe_product_detail_declaratif_distributeur

In [ ]:
# Read paper .xlsx for table of correspondance for product_code, example : Description = MAIZENA + 700G, product_code = MAIZENAPETIT

In [ ]:
# Delete "DECLARATIF DISTRIBUTEUR" in the column "data_source"
dataframe_product_detail_declaratif_distributeur["data_source"] = dataframe_product_detail_declaratif_distributeur["data_source"].str.replace("DECLARATIF DISTRIBUTEUR", "")
dataframe_product_detail_declaratif_distributeur

In [ ]:
dataframe_product_detail_declaratif_distributeur["product_code"] = dataframe_product_detail_declaratif_distributeur.apply(
    lambda row: "to_be_completed" if pd.isnull(row["brand"]) or row["brand"] == "" or row["brand"] == "." else row["brand"],
    axis=1
)

dataframe_product_detail_declaratif_distributeur

In [ ]:
# Clear brand
dataframe_product_detail_declaratif_distributeur["brand"] = ""
dataframe_product_detail_declaratif_distributeur

In [ ]:
# Concatenate the two dataframes
dataframe_product_detail_final = pd.concat([dataframe_product_detail_filtered, dataframe_product_detail_declaratif_distributeur], ignore_index=True)
dataframe_product_detail_final

In [ ]:
dataframe_product_detail_final["amount_ht"] = dataframe_product_detail_final["amount_ht"].round(2)
dataframe_product_detail_final

In [ ]:
# Export in .xlsx
dataframe_product_detail_final.to_excel(
    os.path.join(
        root_path,
        "data_folder",
        "product_detail_export_cleaned.xlsx",
    ),
    index=False
)